In [ ]:
# import needed lib
import kagglehub
import os
import pandas as pd
import numpy as np
import plotly.express as px

state = 42

In [ ]:



# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
q1_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(q1_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
y_dist = px.histogram(
    df,
    x="Delivery_Time"
)
y_dist.show()

In [ ]:
# Task 1: Write your code here:
df.drop("Order_ID", axis=1, inplace= True)

# confirm
df.columns

In [ ]:
# Task 2: Write your code here:

# cheack nulls
display(df.isnull().sum().sort_values(ascending=False))

In [ ]:
# clooser look at ctagoral ones
df["Time_of_Day"].value_counts(), df["Weather"].value_counts(), df["Traffic_Level"].value_counts()

In [ ]:
# clooser look at numaric ones
df[["Delivery_Time", "Courier_Experience_yrs"]].describe()

In [ ]:
# Actions
# for ctagoral :
#   fillna with "Unkown"
#   except the 'Weather' with "Clear" (assume it in saudi + 700 compare to 100 is a huge)
# for numaric :
#  'Courier_Experience_yrs' fillna with mean
#  but for the 'Delivery_Time' dropna since we cant fill it with approxmation it will heart our model very much



df["Time_of_Day"] = df["Time_of_Day"].fillna("Unkown")
df["Traffic_Level"] = df["Traffic_Level"].fillna("Unkown")
df["Weather"] = df["Weather"].fillna(df["Weather"].mode()[0])


df["Courier_Experience_yrs"] = df["Courier_Experience_yrs"].fillna(df["Courier_Experience_yrs"].mean())

df = df.dropna(subset=["Delivery_Time"])




In [ ]:
# confirm
display(df.isnull().sum().sort_values(ascending=False))

In [ ]:
df["Time_of_Day"].value_counts(), df["Weather"].value_counts(), df["Traffic_Level"].value_counts()

In [ ]:
df[["Delivery_Time", "Courier_Experience_yrs"]].describe()

In [ ]:
# Task 3: Write your code here:

# cheack
df.duplicated().sum()

In [ ]:
# action drop them
df = df.drop_duplicates()

In [ ]:
# confirm
df.duplicated().sum()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False)

In [ ]:
# Weather
one_hot = onehot_encoder.fit_transform(df[["Weather"]]) # save the array of one hot

print('\ncol befor encoding:\n', df["Weather"]) # get the befor shape
for i in range(df["Weather"].nunique()) :
  df[f"Weather_{i}"] = one_hot[:,i] # assign each appropriate one hot columns with numbered column in df



df[[f"Weather_{i}"for i in range(df["Weather"].nunique())]] # show the new columns



In [ ]:
df = df.drop("Weather", axis=1)

In [ ]:
# Traffic_Level
one_hot = onehot_encoder.fit_transform(df[["Traffic_Level"]]) # save the array of one hot

print('\ncol befor encoding:\n', df["Traffic_Level"]) # get the befor shape
for i in range(df["Traffic_Level"].nunique()) :
  df[f"Traffic_Level_{i}"] = one_hot[:,i] # assign each appropriate one hot columns with numbered column in df



df[[f"Traffic_Level_{i}"for i in range(df["Traffic_Level"].nunique())]] # show the new columns


In [ ]:
df = df.drop("Traffic_Level", axis=1)

In [ ]:
# Time_of_Day
one_hot = onehot_encoder.fit_transform(df[["Time_of_Day"]]) # save the array of one hot

print('\ncol befor encoding:\n', df["Time_of_Day"]) # get the befor shape
for i in range(df["Time_of_Day"].nunique()) :
  df[f"Time_of_Day_{i}"] = one_hot[:,i] # assign each appropriate one hot columns with numbered column in df



df[[f"Time_of_Day_{i}"for i in range(df["Time_of_Day"].nunique())]] # show the new columns



In [ ]:

df = df.drop("Time_of_Day", axis=1)

In [ ]:


# Vehicle_Type
one_hot = onehot_encoder.fit_transform(df[["Vehicle_Type"]]) # save the array of one hot

print('\ncol befor encoding:\n', df["Vehicle_Type"]) # get the befor shape
for i in range(df["Vehicle_Type"].nunique()) :
  df[f"Vehicle_Type_{i}"] = one_hot[:,i] # assign each appropriate one hot columns with numbered column in df



df[[f"Vehicle_Type_{i}"for i in range(df["Vehicle_Type"].nunique())]] # show the new columns




In [ ]:
df = df.drop("Vehicle_Type", axis=1)

In [ ]:
# confirm
df.head()

In [ ]:
# Task 5: Write your code here:

# split the target and the features before scalling (so we dont scale the target)

features = [col for col in df.columns if col != 'Delivery_Time']

X = df[features]
y = df["Delivery_Time"]

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)

In [ ]:
# confirm before and after
print(df.head().values) # before
print(df_scaled[0:5]) # after

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:

from sklearn.model_selection import train_test_split

# split ratio (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% for test, remaining 80% for train
    random_state=state,      # reproducible output
    shuffle=True          # representative splits
)
# Print shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(random_state=state, n_estimators=200)


In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)


maes = []

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Training
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # evaluation matrecs
    mae = mean_absolute_error(y_test, y_pred)

    maes.append(mae)

maes = np.array(maes)
avg_mse = maes.mean()
print(avg_mse)




In [ ]:
# Task 1: Write your code here:

import matplotlib.pyplot as plt

importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
})
importance = importance.sort_values('importance', ascending=True).tail(10)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.title('Top 10 Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

# Y_Hat
plt.figure(figsize=(10, 6))
plt.hist(y_pred)

plt.title('predected y_hat distrbuation')

plt.tight_layout()
plt.show()

In [ ]:
# THE ORIGIONAL Y
y_dist.show()

In [ ]:
# Task Bonus: Write your code here:

# why not try another state ?? I will do it  :)

state_1, state_2 = 220,112


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model_1 = RandomForestRegressor(random_state=state_1, n_estimators=200)
model_2 = RandomForestRegressor(random_state=state_2, n_estimators=200)


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

maes = []

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Training
    model_1.fit(X_train, y_train)
    model_2.fit(X_train, y_train)

    # Predict
    y_pred_1 = model_1.predict(X_test)
    y_pred_2 = model_2.predict(X_test)


    avg_y_hat = (y_pred_1 + y_pred_2) / 2

    # evaluation matrecs
    mae = mean_absolute_error(y_test, avg_y_hat)

    maes.append(mae)

maes = np.array(maes)
avg_mse = maes.mean()
print(avg_mse)

In [ ]:
# not much diffrent (maybe worest than the singal one) but first time i think in that way when building a model
print("DONE!!")